In [1]:
from pathlib import Path
import re
import pandas as pd

FOLDER = Path(".")                     # folder that contains the raw files
OUTPUT = FOLDER / "preprocessed"       # where the clean tables are saved
OUTPUT.mkdir(exist_ok=True)

report = []                            # every message is also saved to a report file


def log(text=""):
    print(text)
    report.append(str(text))


# February uses different names for some header columns
RENAME = {
    "mbr_id": "member_id",
    "svc_from_dt": "service_from_date",
    "svc_to_dt": "service_to_date",
    "allowed_amt": "total_allowed",
    "paid_amt": "total_paid",
}
HEADER_DATES = ["service_from_date", "service_to_date", "received_date", "processed_date"]
HEADER_MONEY = ["total_billed", "total_allowed", "total_paid", "member_responsibility"]
LINE_MONEY = ["billed_amount", "allowed_amount", "paid_amount"]

## Helper functions

In [2]:
def find_files(prefix):
    """Return {month: file} for raw files only. Month looks like '2026_01'."""
    found = {}
    for f in sorted(FOLDER.rglob(f"{prefix}_*.csv")):
        if ("_cleaned" in f.name or "checkpoint" in f.name
                or ".ipynb_checkpoints" in f.parts
                or "cleaned_data" in f.parts or "preprocessed" in f.parts):
            continue
        month = re.search(r"(\d{4}_\d{2})", f.name).group(1)
        if month in found:
            log(f"WARNING: second file for {prefix} {month} ignored: {f}")
            continue
        found[month] = f
    return found


def parse_dates(series):
    """Convert each value using its own format, so mixed formats are safe.
    YYYY-MM-DD stays as is. DD-MM-YYYY is read day-first (08-04-2025 = 8 April)."""
    s = series.astype("string")
    iso = s.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
    dmy = s.str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    out[iso] = pd.to_datetime(s[iso], format="%Y-%m-%d")
    out[dmy] = pd.to_datetime(s[dmy], format="%d-%m-%Y")
    bad = s.notna() & out.isna()
    if bad.any():
        log(f"   WARNING: {int(bad.sum())} date values could not be read")
    return out


def to_number(series, name):
    num = pd.to_numeric(series, errors="coerce")
    bad = series.notna() & num.isna()
    if bad.any():
        log(f"   WARNING: {int(bad.sum())} values in {name} are not numbers")
    return num


def fix_code(value):
    """Procedure codes: 110 -> 0110 (digits only and shorter than 4)."""
    if pd.notna(value) and str(value).isdigit() and len(str(value)) < 4:
        return str(value).zfill(4)
    return value

## Step 1: Find the raw files

In [3]:
log("STEP 1: Finding raw files")
header_files = find_files("claims_header")
line_files = find_files("claim_lines")
premium_files = find_files("premiums")
for name, files in [("claims header", header_files), ("claim lines", line_files),
                    ("premiums", premium_files)]:
    log(f"   {name}: {list(files.keys())}")
if not header_files:
    raise FileNotFoundError("No raw files found. Run this from the folder with the Feed_... folders.")

STEP 1: Finding raw files
   claims header: ['2026_01', '2026_02', '2026_03']
   claim lines: ['2026_01', '2026_02', '2026_03']
   premiums: ['2026_01', '2026_02', '2026_03']


## Step 2: Claims header - standardise and clean

In [4]:
log("\nSTEP 2: Claims header - standardise and clean")
parts = []
for month, f in header_files.items():
    df = pd.read_csv(f, dtype="string")
    df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    df = df.rename(columns=RENAME)
    for col in df.columns:                       # trim spaces, blank -> missing
        df[col] = df[col].str.strip().replace("", pd.NA)
    df["feed_month"] = month
    parts.append(df)
    log(f"   {month}: {len(df)} rows, {df.shape[1] - 1} columns")

# telehealth_flag only exists in March; concat fills the other months with missing values
headers = pd.concat(parts, ignore_index=True)

for col in HEADER_DATES:
    headers[col] = parse_dates(headers[col])
for col in HEADER_MONEY:
    headers[col] = to_number(headers[col], col)

data_cols = [c for c in headers.columns if c != "feed_month"]
exact = int(headers.duplicated(data_cols).sum())
headers = headers.drop_duplicates(data_cols)
log(f"   exact duplicate rows removed: {exact}")


STEP 2: Claims header - standardise and clean
   2026_01: 6404 rows, 19 columns
   2026_02: 5266 rows, 19 columns
   2026_03: 4972 rows, 20 columns
   exact duplicate rows removed: 35


## Step 3: Keep the latest version of each claim

In [5]:
log("\nSTEP 3: Keep the latest version of each claim")
rows_before = len(headers)
version_count = headers.groupby("claim_id").size().rename("version_count")
first_seen = headers.groupby("claim_id").feed_month.min().rename("first_seen_feed")
claims = (headers.sort_values("feed_month", kind="stable")
                 .drop_duplicates("claim_id", keep="last")
                 .merge(version_count, on="claim_id")
                 .merge(first_seen, on="claim_id"))
log(f"   header rows {rows_before} -> unique claims {len(claims)} "
    f"({rows_before - len(claims)} older versions removed)")
log(f"   claim status counts: {claims.claim_status.value_counts().to_dict()}")


STEP 3: Keep the latest version of each claim
   header rows 16607 -> unique claims 14462 (2145 older versions removed)
   claim status counts: {'Paid': 12433, 'Denied': 1072, 'Pending': 866, 'Adjusted': 91}


## Step 4: Helper columns for the fraud analysis

In [6]:
log("\nSTEP 4: Helper columns for the fraud analysis")
claims["is_adjustment"] = claims.claim_status.eq("Adjusted")
claims["days_to_submit"] = (claims.received_date - claims.service_from_date).dt.days
claims["processing_days"] = (claims.processed_date - claims.received_date).dt.days
claims["received_before_service"] = claims.received_date < claims.service_from_date
log(f"   claims received before the service date: {int(claims.received_before_service.sum())}")


STEP 4: Helper columns for the fraud analysis
   claims received before the service date: 30


## Step 5: Claim lines - clean and keep the latest version

In [7]:
log("\nSTEP 5: Claim lines - clean and keep the latest version")
parts = []
for month, f in line_files.items():
    df = pd.read_csv(f, dtype="string")
    df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    for col in df.columns:
        df[col] = df[col].str.strip().replace("", pd.NA)
    df["feed_month"] = month
    parts.append(df)
lines = pd.concat(parts, ignore_index=True)
lines["procedure_code"] = lines.procedure_code.map(fix_code)
lines["line_number"] = to_number(lines.line_number, "line_number").astype("Int64")
lines["units"] = to_number(lines.units, "units")
for col in LINE_MONEY:
    lines[col] = to_number(lines[col], col)

line_cols = [c for c in lines.columns if c != "feed_month"]
lines = lines.drop_duplicates(line_cols)
rows_before = len(lines)
# a claim's lines come from the LAST feed in which that claim has lines
last_feed = lines.groupby("claim_id").feed_month.transform("max")
lines = lines[lines.feed_month == last_feed].copy()
log(f"   line rows {rows_before} -> {len(lines)} after keeping the latest version")


STEP 5: Claim lines - clean and keep the latest version
   line rows 31765 -> 27673 after keeping the latest version


## Step 6: Premiums

In [8]:
log("\nSTEP 6: Premiums")
parts = []
for month, f in premium_files.items():
    df = pd.read_csv(f, dtype="string")
    df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    df["feed_month"] = month
    parts.append(df)
premiums = pd.concat(parts, ignore_index=True)
premiums["premium_amount"] = to_number(premiums.premium_amount, "premium_amount")
n = len(premiums)
premiums = premiums.drop_duplicates(["member_id", "premium_month"], keep="last")
log(f"   premium rows {n} -> {len(premiums)} (one row per member per month)")


STEP 6: Premiums
   premium rows 20784 -> 20784 (one row per member per month)


## Step 7: Checks and reference totals

In [9]:
log("\nSTEP 7: Checks on the clean tables")


def check(ok, text):
    log(("   PASS  " if ok else "   FAIL  ") + text)


check(claims.claim_id.is_unique, "claim_id is unique in the claims table")
check(not lines.duplicated(["claim_id", "line_number"]).any(), "claim_id + line_number is unique in the lines table")
check(lines.claim_id.isin(claims.claim_id).all(), "every claim line has a claim")
check(claims.claim_id.isin(lines.claim_id).all(), "every claim has claim lines")
check(not premiums.duplicated(["member_id", "premium_month"]).any(), "one premium row per member per month")

sums = lines.groupby("claim_id")[["billed_amount", "paid_amount"]].sum()
j = claims.set_index("claim_id").join(sums, how="inner")
check(((j.total_billed - j.billed_amount).abs() > 0.05).sum() == 0, "claim total billed = sum of its lines")
check(((j.total_paid - j.paid_amount).abs() > 0.05).sum() == 0, "claim total paid = sum of its lines")

# Reference totals. If another tool shows different numbers, one of us has a bug.
log("\nREFERENCE TOTALS (these must match Intern 1's SQL tables)")
log(f"   unique claims        : {len(claims)}")
log(f"   claim lines          : {len(lines)}")
log(f"   premium rows         : {len(premiums)}")
log(f"   total billed         : {claims.total_billed.sum():,.2f}")
log(f"   total allowed        : {claims.total_allowed.sum():,.2f}")
log(f"   total paid           : {claims.total_paid.sum():,.2f}")
log(f"   total premiums       : {premiums.premium_amount.sum():,.2f}")


STEP 7: Checks on the clean tables
   PASS  claim_id is unique in the claims table
   PASS  claim_id + line_number is unique in the lines table
   PASS  every claim line has a claim
   PASS  every claim has claim lines
   PASS  one premium row per member per month
   PASS  claim total billed = sum of its lines
   PASS  claim total paid = sum of its lines

REFERENCE TOTALS (these must match Intern 1's SQL tables)
   unique claims        : 14462
   claim lines          : 27673
   premium rows         : 20784
   total billed         : 24,609,180.97
   total allowed        : 9,472,303.09
   total paid           : 7,526,594.02
   total premiums       : 11,029,680.00


## Step 8: Save the clean tables

In [10]:
for col in HEADER_DATES:
    claims[col] = claims[col].dt.strftime("%Y-%m-%d")
claims.to_csv(OUTPUT / "claims_clean.csv", index=False, encoding="utf-8")
lines.to_csv(OUTPUT / "claim_lines_clean.csv", index=False, encoding="utf-8")
premiums.to_csv(OUTPUT / "premiums_clean.csv", index=False, encoding="utf-8")
(OUTPUT / "preprocessing_report.txt").write_text("\n".join(report), encoding="utf-8")
log(f"\nSaved to {OUTPUT.resolve()}: claims_clean.csv, claim_lines_clean.csv, "
    "premiums_clean.csv, preprocessing_report.txt")


Saved to C:\Users\munic\Desktop\Oriana Data Analytics Internship\Final Project\Dataset\preprocessed: claims_clean.csv, claim_lines_clean.csv, premiums_clean.csv, preprocessing_report.txt
